# Adquisición de datos de Lídar con tracking.

**Objetivo:** A partir de los datos del lídar en la infraestructura suministrado, aplicar el algoritmo de DBScan para detectar el número de vehículos que hay en la muestra de datos.

A continuación, realizar un tracking de los vehículos que circulan por la carretera, a fin de asegurar la identificación de los vehículos y su seguimiento con mismos IDs en todos los frames.


## 1. Librerías utilizadas

- **pandas**: para cargar el CSV como una tabla (`DataFrame`) y poder filtrar,
  seleccionar columnas y hacer estadísticas descriptivas fácilmente.
- **numpy**: la base numérica de Python científico. La usamos para trabajar con
  los puntos como arrays de números (vectores y matrices), mucho más rápido que
  usar listas de Python normales.
- **matplotlib**: para dibujar gráficas (dispersión de puntos, histogramas) y
  poder inspeccionar visualmente la nube de puntos y los resultados.
- **open3d**: librería especializada en nubes de puntos 3D. La usamos para
  detectar automáticamente el plano del suelo (algoritmo RANSAC), algo tedioso
  de programar a mano desde cero.
- **scikit-learn**: aquí están implementados los algoritmos de clustering que
  pide la práctica, `DBSCAN` y `KMeans`, ya optimizados y listos para usar.
- **scipy**: aquí esta implementada la estructura de búsqueda espacial `KDTree` que
  utilizaremos para la eliminación de puntos no necesarios.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from sklearn.cluster import DBSCAN, KMeans
from scipy.spatial import cKDTree

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Carga y exploración de los datos de cada uno de los frames

Creamos una función que, mediante dbscan, extrae los vehiculos del frame que se le da.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

def procesar_frame_dbscan(filepath, eps_val=1.5, min_samples_val=15):
    # Leer datos limpios
    df = pd.read_csv(filepath)
    coords = df[['x', 'y', 'z']].values
    
    # Aplicar clustering
    modelo = DBSCAN(eps=eps_val, min_samples=min_samples_val)
    labels = modelo.fit_predict(coords)
    
    vehiculos = []
    
    # Extraer características para el Tracking
    for etiqueta in set(labels):
        if etiqueta == -1:
            continue # Descartar ruido atmosférico o errores del sensor
            
        puntos_cluster = coords[labels == etiqueta]
        centroide = np.mean(puntos_cluster, axis=0)
        
        vehiculos.append({
            'cluster_id_local': etiqueta,
            'centroide': centroide,
            'num_puntos': len(puntos_cluster)
        })
        
    return vehiculos

## 3. Tracking de los vehiculos
Seguiremos la siguiente lógica en cada frame para mantener un registro global de los vehículos detectados:
1. Matriz de costes: calculamos la distancia física exacta entre todos los centroides conocidos (frames anteriores) y los nuevos centroides detectados (frame actual).

2. Asignación Óptima: utilizamos scipy.optimize.linear_sum_assignment, emparejamos cada vehículo nuevo con el antiguo que tenga la distancia mínima, minimizando el coste total.

3. Umbral de Distancia Máxima: establecemos un límite entre frames para poder concluir que si un centroide se ha movido más de esa distancia es un objeto nuevo o un error.

4. Gestión del Ciclo de Vida: Si un vehículo desaparece no le asignamos ningún nuevo centroide, se elimina del registro activo después de cierto número de frames de tolerancia.

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

class TrackerVehiculos:
    def __init__(self, distancia_maxima_movimiento=3.0, tolerancia_frames_perdidos=2):
        self.distancia_max = distancia_maxima_movimiento
        self.tolerancia_perdida = tolerancia_frames_perdidos
        self.siguiente_id = 1
        
        # Diccionario para almacenar el estado actual de los vehículos
        # Clave: ID global, Valor: {'centroide': array, 'frames_perdido': int}
        self.vehiculos_activos = {}

    def actualizar(self, detecciones_nuevo_frame):
        """
        Recibe una lista de diccionarios (los vehículos detectados por DBSCAN)
        y devuelve la misma lista pero con los IDs globales asignados.
        """
        # Si no hay vehículos guardados, registramos todos como nuevos
        if len(self.vehiculos_activos) == 0:
            for det in detecciones_nuevo_frame:
                det['track_id'] = self.siguiente_id
                self.vehiculos_activos[self.siguiente_id] = {
                    'centroide': det['centroide'],
                    'frames_perdido': 0
                }
                self.siguiente_id += 1
            return detecciones_nuevo_frame

        # Extraer listas ordenadas de IDs activos y sus centroides
        ids_activos = list(self.vehiculos_activos.keys())
        centroides_antiguos = np.array([self.vehiculos_activos[i]['centroide'] for i in ids_activos])
        
        # Extraer centroides detectados en el frame actual
        centroides_nuevos = np.array([d['centroide'] for d in detecciones_nuevo_frame])

        if len(centroides_nuevos) == 0:
            # Si no hay detecciones, sumar a los contadores de pérdida
            self._incrementar_perdidas(ids_activos)
            return []

        # 1. Calcular matriz de distancias (Costes)
        # Tamaño de la matriz: (N_antiguos, M_nuevos)
        matriz_distancias = cdist(centroides_antiguos, centroides_nuevos)

        # 2. Resolver la asignación óptima (Algoritmo Húngaro)
        indices_antiguos, indices_nuevos = linear_sum_assignment(matriz_distancias)

        # 3. Filtrar emparejamientos válidos usando el umbral de distancia máxima
        asignaciones_validas = []
        indices_nuevos_asignados = set()
        indices_antiguos_asignados = set()

        for idx_antiguo, idx_nuevo in zip(indices_antiguos, indices_nuevos):
            distancia = matriz_distancias[idx_antiguo, idx_nuevo]
            
            if distancia <= self.distancia_max:
                asignaciones_validas.append((idx_antiguo, idx_nuevo))
                indices_nuevos_asignados.add(idx_nuevo)
                indices_antiguos_asignados.add(idx_antiguo)
                
                # Actualizar el registro del vehículo con la nueva posición
                id_global = ids_activos[idx_antiguo]
                self.vehiculos_activos[id_global]['centroide'] = centroides_nuevos[idx_nuevo]
                self.vehiculos_activos[id_global]['frames_perdido'] = 0
                
                # Asignar el ID global al objeto devuelto
                detecciones_nuevo_frame[idx_nuevo]['track_id'] = id_global

        # 4. Registrar detecciones nuevas (sin emparejar)
        indices_nuevos_no_asignados = set(range(len(centroides_nuevos))) - indices_nuevos_asignados
        for idx in indices_nuevos_no_asignados:
            det = detecciones_nuevo_frame[idx]
            det['track_id'] = self.siguiente_id
            self.vehiculos_activos[self.siguiente_id] = {
                'centroide': det['centroide'],
                'frames_perdido': 0
            }
            self.siguiente_id += 1

        # 5. Gestionar vehículos perdidos
        indices_antiguos_no_asignados = set(range(len(centroides_antiguos))) - indices_antiguos_asignados
        ids_perdidos = [ids_activos[idx] for idx in indices_antiguos_no_asignados]
        self._incrementar_perdidas(ids_perdidos)

        return detecciones_nuevo_frame

    def _incrementar_perdidas(self, ids_perdidos):
        # Incrementa el contador de frames no detectado y elimina si supera tolerancia
        ids_a_borrar = []
        for track_id in ids_perdidos:
            self.vehiculos_activos[track_id]['frames_perdido'] += 1
            if self.vehiculos_activos[track_id]['frames_perdido'] > self.tolerancia_perdida:
                ids_a_borrar.append(track_id)
                
        for track_id in ids_a_borrar:
            del self.vehiculos_activos[track_id]

Número de puntos: 131072
Columnas: ['x', 'y', 'z', 'intensity', 't', 'reflectivity', 'ring', 'ambient', 'range']
<class 'pandas.DataFrame'>
RangeIndex: 131072 entries, 0 to 131071
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   x             131072 non-null  float64
 1   y             131072 non-null  float64
 2   z             131072 non-null  float64
 3   intensity     131072 non-null  float64
 4   t             131072 non-null  int64  
 5   reflectivity  131072 non-null  int64  
 6   ring          131072 non-null  int64  
 7   ambient       131072 non-null  int64  
 8   range         131072 non-null  int64  
dtypes: float64(4), int64(5)
memory usage: 9.0 MB


In [ ]:
import glob
import os

# 1. Obtener la lista de los 300 CSVs ordenada por tiempo
lista_archivos = sorted(glob.glob("ruta/a/tus/archivos/pointcloud_*.csv"))

# 2. Inicializar el tracker
tracker = TrackerVehiculos(distancia_maxima_movimiento=3.0)

resultados_tracking = []

# 3. Iterar por todos los frames
for archivo in lista_archivos:
    # Fase 1: Detectar vehículos en este frame específico
    vehiculos_detectados = procesar_frame_dbscan(archivo, eps_val=1.5, min_samples_val=15)
    
    # Fase 2: Actualizar el tracker para asignar IDs consistentes
    vehiculos_rastreados = tracker.actualizar(vehiculos_detectados)
    
    # Guardar resultados o preparar para visualización
    for vehiculo in vehiculos_rastreados:
        vehiculo['frame'] = os.path.basename(archivo)
        resultados_tracking.append(vehiculo)

print(f"Tracking completado. Vehículos totales únicos detectados: {tracker.siguiente_id - 1}")